###### Content under Creative Commons Attribution license CC-BY 4.0, code under BSD 3-Clause License © 2022  by D. Koehn, T. Meier and J. Stampa, notebook style sheet by L.A. Barba, N.C. Clementi

# Digital Signal Processing in Geophysics 

## Chapter 5: Signal Stacking

In stacking, a model of the form

\begin{equation}
x_i(t) = s(t - t_i) + n_i(t) \tag{5.1}
\end{equation}

is generally assumed. Here, the sequence $x_i$ consists of a deterministic signal $s(t)$ that is time-shifted by $-t_i$ in the $i$th sequence. It is assumed that the form of the signal is the same across the different sequences. $n_i(t)$ is random noise, which is often assumed to be a Gaussian iid random process with $E[n_i(t)] = 0$. 
In stacking, $M$ measured time series are shifted in time by estimates $\hat{t}_i$ and averaged:

\begin {equation}
\mbox{stack}(t) = \frac{1}{M} \sum_{i=1}^M x_i(t+\hat{t}_i). \tag{5.2}
\end{equation}

The aim is to reverse the time shift. The time series are then averaged. The estimates $\hat{t}_i$ are calculated for a hypothesis to be tested. If it is correct or very close to the actual situation, constructive interference is present and $\mbox{stack}(t)$ takes on large values. Low values of $\mbox{stack}(t)$, on the other hand, indicate destructive interference, and the tested hypothesis should rather be rejected. In the case where $\hat{t}_i = t_i$, the following holds:  

\begin{equation}
E[\mbox{stack}(t)] = s(t) \tag{5.3}
\end{equation}

and for large $M$, $\mbox{stack}(t)$ takes on the form of the signal.

I will illustrate the purpose of Eq. (5.2) using a simple example. First, we import some Python libraries ...

In [ ]:
# Import Python Libraries 
# -----------------------
#%matplotlib inline
from ipywidgets import interactive
import matplotlib.pyplot as plt
import numpy as np
from gsv.gsv_func import *        # library "Geophysical Signal Processing"

... including a new library called “Geophysical Signal Processing”, which contains various functions that I wrote specifically for the lecture. The function `slant_stack_intro` calculates a highly simplified shot gather and stacks the seismograms ...

In [ ]:
interactive_plot = interactive(slant_stack_intro, vstack=(10., 400., 10.), stackcorr=False )
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

As can be seen, the shot gather contains only a single direct SH wave (left). Simply stacking all tracks over the offsets (right) does not make sense, since we are not summing the wave signals but rather time-shifted tracks. According to Eq. (5.2), we must first correct the travel time difference 

\begin{equation}
\hat{t} = - \frac{\text{offset}}{vs}\notag
\end{equation}

where $\text{offset}$ denotes the offset of the geophones from the source position and $vs$ denotes the S-wave velocity of the subsurface. We can activate this travel time correction using the **stackcorr** option and vary the stacking velocity with the **vstack** parameter. For a stacking velocity of vstack = 200 m/s, the travel time difference is perfectly compensated, and constructive interference leads to an amplification of the signal. As described in Eq. (5.3), the stack determines the shape of the signal. 

We can also interpret the combination of time shift and stack as summing the waveform along the direct wave’s travel time curve. If vstack is greater than or less than the optimal stacking velocity, destructive interference occurs and the stacked amplitudes become correspondingly smaller. We can use this to develop a simple tool to determine the S-wave velocity of the subsurface based on the stacking along travel time curves. 

Before we take a closer look at this, we will use some simple examples to illustrate a few properties of stacking. 

### 5.1 Examples

In the simplest case, $\hat{t}_i = 0$. The stack is then:

\begin{equation}
\mbox{stack}(t) = \frac{1}{M} \sum_{i=1}^M x_i(t). \tag{5.4}
\end{equation}

For example, in the case of hammer-shot seismic surveys, the signal-to-noise ratio can be improved through repeated excitation. The expected value for the stack is:

\begin{equation}
E[\mbox{stack}(t)] = s(t). \tag{5.5}
\end{equation}

As $M$ increases, the noise is suppressed more strongly. The signal-to-noise ratio after $M$ measurements can be measured according to Section 2.3 (Signal-to-Noise Ratio) as 

\begin{equation}
\frac{s_{max}}{\sigma_n(M)}. \tag{5.6}
\end {equation}

Here, $s_{max}$ is the maximum amplitude of the signal, and $\sigma_n(M)$ is the expected deviation of the stacked noise from the expected value of the noise after averaging over $M$ measurements. The expected value of the noise is zero, and the standard deviation of the noise is $\sigma_n$. Since for $\sigma_n(M)$, the error in the estimate of the expected value after averaging over $M$ measurements, or the standard error,

\begin{equation}
\sigma_n(M) = \frac{\sigma_n} {\sqrt{M}} \tag{5.7}
\end{equation} 

it is expected that the signal-to-noise ratio improves proportionally to $\sqrt{M}$:

\begin{equation}
\frac{s_{max} \sqrt{M}}{\sigma_n}. \tag{5.8}
\end{equation}

Let's test this concept using a sine function to which we add normally distributed noise and stack M time series. 

In [ ]:
def stack(sigman, M):
    
    # Define parameters
    T = 25.                  # period 1 [s]
    dt = .1                  # time sampling [s]
    L = 500.                # length of the time series [s]

    # Define sine function ...
    omega = 2. * np.pi / T   # compute circular frequency from period
    t = np.arange(0,L+dt,dt) # compute time vector
    x = np.sin(omega*t)  # compute sine wave
        
    # Add normal distributed noise to the time series
    mu = 0.
    
    # Initialize stack
    N = len(x)
    xstack = np.zeros(N)
    for i in range(M):
    
        # Add random noise to time series
        xnoise = np.random.normal(mu, sigman, N) # create random numbers
        x1 = x + xnoise # add random noise to time series 
        
        # Stack time series
        xstack = xstack + x1 
        
    # Divide xstack by number of stacks M
    xstack = xstack / M
        
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    plt.plot(t, xstack, 'b')
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')
    
    # Estimate and print S/N ratio
    SN = np.sqrt(M) / sigman 
    title = r'Stacked Time Series f(t) = Sin(t) ($S/N = \frac{smax\sqrt{M}}{\sigma_n} = $' + str(SN) + ')'
    
    plt.title(title)

We can define the noise amplitude using the standard deviation of the normally distributed random numbers $sigman$ and the size of the stack, $M$ ...

In [ ]:
interactive_plot = interactive(stack, sigman=(0., 10., .5), M=(1, 10000, 10) )
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

So far, we investigated what the impact of the stacking operation on the noise is. Next, let 's investigate the effect of stacking operations on different signals ...

# Example 1:
Superposition of two signals with the same frequency $\omega_0$ but different phases. This is a simple example of a superposition with $t_i \neq 0$ and $n_i = 0$. The two signals can be described by:

\begin{align}
s_1(t) & = \cos(\omega_0 t + \varphi_1),\tag{5.9}\\
s_2(t) & =\cos(\omega_0 t+\varphi_2)\tag{5.10}
\end{align}

Does the superposition change the frequency contained in the signal? What does the amplitude of the result depend on?
According to the addition theorem, the superposition of the two signals yields:

\begin{equation}
s_1(t)+s_2(t)=2 \cos \left(\omega_0 t + \frac {\varphi_1 + \varphi_2}{2}\right)\, \cos\left(\frac {\varphi_1 - \varphi_2}{2}\right). \tag{5.11}
\end{equation}

This means that the frequency $\omega_0$ is preserved after superposition. The frequencies present in the signal do not change as a result of superposition. The phase shift is averaged. The amplitude depends on the phase difference, since the time-independent weighting factor depends on the phase difference.  

Let’s take a closer look at this ...

In [ ]:
def cos_phase(phi1, phi2):
    
    # Define parameters
    T = 60.                  # period 1 [s]
    dt = .1                  # time sampling [s]
    L = 200.                # length of the time series [s]

    # Define sine function ...
    omega = 2. * np.pi / T        # compute circular frequency from period
    t = np.arange(0,L+dt,dt)      # compute time vector
    s1 = np.cos(omega*t + phi1)   # compute sine wave 1
    s2 = np.cos(omega*t + phi2)   # compute sine wave 2
        
    # Initialize Plots
    plt.figure(figsize=(10,5))
    
    # Plot time series
    plt.plot(t, s1, 'b',label=r'$s_1(t) = \cos(\omega_0 t+\varphi_1)$')
    plt.plot(t, s2, 'c',label=r'$s_2(t) = \cos(\omega_0 t+\varphi_2)$')
    plt.plot(t, s1+s2, 'r',label=r'$s_1 +s_2$')
    plt.legend()
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')

    title = r'Sum of Time Series $s_1(t) + s_2(t)$ ($\Delta \phi$ =' + str((phi1-phi2)/np.pi) + ' $\pi$)'    
    plt.title(title)

In [ ]:
interactive_plot = interactive(cos_phase, phi1=(0.,2*np.pi,np.pi/4.), phi2=(0.,2*np.pi,np.pi/4.) )
output = interactive_plot.children[-1]
output.layout.height = '500px'
interactive_plot

For a phase shift of $|\varphi_1-\varphi_2|\le \frac{\pi}{2}$ (which corresponds to a phase shift of one-quarter of the period or, in the local region, one-quarter of the wavelength), the weighting factor $\cos \left(\frac{\varphi_1-\varphi_2}{2} \right)\ge 0.7071$ holds, and thus constructive interference occurs. For increasingly larger phase differences, the amplitude of the resulting signal decreases. This indicates destructive interference. For $\varphi_1-\varphi_2= \pi$, the signals cancel each other out. For phase differences greater than $\pi$, the amplitude increases again due to the periodicity of the signals.

# Example 2

Next, we consider the sum of two signals with different frequencies:

\begin{align}
s_1(t) & = \cos(\omega_1 t),\tag{5.12}\\
s_2(t) &= \cos(\omega_2 t),\tag{5.13}
\end{align}

where $\omega_1 > \omega_2$. By the addition theorem, we have:

\begin{equation}
s_1(t)+s_2(t)= 2 \cos \left(\frac{\omega_1+\omega_2}{2}t\right)\, \cos \left(\frac{\omega_1-\omega_2}{2}t\right).\tag{5.14}
\end{equation}

The result is a signal containing a high-frequency carrier circular frequency $\frac{\omega_1+\omega_2}{2}$, which corresponds to the center circular frequency, and whose amplitude is modulated at a low circular frequency with $\frac{\omega_1-\omega_2}{2}$ or with the frequency $f_A = \frac{1}{2\pi}\frac{\omega_1-\omega_2}{2}$.

In [ ]:
def cos_freq(T1, T2):
    
    # Define parameters
    dt = .1                  # time sampling [s]
    L = 200.                # length of the time series [s]

    # Define sine function ...
    t = np.arange(-L,L+dt,dt)      # compute time vector
    
    omega1 = 2. * np.pi / T1      # compute circular frequency from period 1
    omega2 = 2. * np.pi / T2      # compute circular frequency from period 2
    
    s1 = np.cos(omega1*t)   # compute sine wave 1
    s2 = np.cos(omega2*t)   # compute sine wave 2
    
    # add both sine waves
    s = s1 + s2
    
    # Fourier transform
    S = np.fft.fft(s)
    
    # Normalize by length of the time series
    N = len(s)               # length of the time series [#samples]
    S = S / (N*dt)
    
    # estimate frequencies    
    freq = np.fft.fftfreq(N, d=dt)
    
    # Initialize Plots
    plt.figure(figsize=(20,10))
    
    # Plot time series
    plt.subplot(211)
    
    plt.plot(t, s1, 'b',label=r'$s_1(t) = \cos(\omega_1 t)$')
    plt.plot(t, s2, 'c',label=r'$s_2(t) = \cos(\omega_2 t)$')
    plt.plot(t, s1+s2, 'r',label=r'$s_1 +s_2$')
    plt.legend()
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')
    plt.xlim(-200,200)    
    plt.title(r'Sum of Time Series $s_1(t) + s_2(t)$')
    
    # Plot Spectrum
    plt.subplot(212)
    
    plt.plot(freq, np.abs(S), 'b')
    plt.xlabel('Freq (Hz)')
    plt.ylabel('Amplitude |X(freq)|')
    plt.title(r'Amplitude spectrum of Time Series $s_1(t) + s_2(t)$')
    plt.xlim(0., .125)

In [ ]:
interactive_plot = interactive(cos_freq, T1=(0.,100.,1.), T2=(0.,120.,1.) )
output = interactive_plot.children[-1]
output.layout.height = '550px'
interactive_plot

It is interesting to note that when $\frac{\omega_1-\omega_2}{2}t = \frac{\pi}{2} + n\pi$ with $n \in \mathbb{G}$, the sign of the amplitude modulation changes. This results in a phase jump in the resulting signal at the times $\frac{1}{4f_A}+\frac{n}{2f_A}$. In this example ($T_1$=50 s, $T_2$=60 s), this occurs at $\pm 150 s$: successive phases have the same polarity. The maxima of the envelopes are separated by $\frac{1}{2f_A}$. If the frequencies are similar, so-called beats occur, e.g., for $T_1$=11 s, $T_2$=12 s. 

Note: If a Fourier analysis is performed on this signal, only the frequencies $\omega_1$ and $\omega_2$ appear, and not the average carrier frequency or the frequency of the amplitude modulation. The formulation using the amplitude-modulated carrier frequency is mathematically equivalent to the sum of the two signals. Since in Fourier analysis a function is represented by the sum of cosine functions of different frequencies, the superimposed frequencies are detected.

# Example 3

In the next example, a signal is to contain two frequencies $\omega_1$ and $\omega_2$, where $\omega_1 > \omega_2$: $s(t) = \cos(\omega_1t) + \cos(\omega_2t)$. We consider the superposition of this signal with itself, where the second signal is time-shifted by $\tau$ relative to the first: 

\begin{align}
s_1(t) &=s(t),\tag{5.15}\\
s_2(t) &=s(t+\tau).\tag{5.16}
\end{align}

In [ ]:
def cos_freq_tau(T1, T2,tau):
    
    # Define parameters
    dt = .1                  # time sampling [s]
    L = 180.                 # length of the time series [s]

    # Define sine function ...
    t = np.arange(0.,L+dt,dt)      # compute time vector
    
    omega1 = 2. * np.pi / T1      # compute circular frequency from period 1
    omega2 = 2. * np.pi / T2      # compute circular frequency from period 2
    
    s1 = np.cos(omega1*t) + np.cos(omega2*t) # compute sine wave 1
    s2 = np.cos(omega1*(t+tau)) + np.cos(omega2*(t+tau))   # compute sine wave 2
    
    s = s1 + s2
    
    # Fourier transform
    S = np.fft.fft(s)
    
    # Normalize by length of the time series
    N = len(s)               # length of the time series [#samples]
    S = S / (N*dt)
    
    # estimate frequencies    
    freq = np.fft.fftfreq(N, d=dt)
    
    # Initialize Plots
    plt.figure(figsize=(20,10))
    
    # Plot time series
    plt.subplot(211)
    
    plt.plot(t, s1, 'b',label=r'$s_1(t) = \cos(\omega_1 t) + \cos(\omega_2 t)$')
    plt.plot(t, s2, 'c',label=r'$s_2(t) = \cos(\omega_1 (t+\tau)) + \cos(\omega_2 (t+\tau))$')
    plt.plot(t, s, 'r',label=r'$s_1 +s_2$')
    plt.legend()
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')
    plt.xlim(0.,180)    
    plt.title(r'Sum of Time Series $s_1(t) + s_2(t)$')
    
    # Plot Spectrum
    plt.subplot(212)
    
    plt.plot(freq, np.abs(S), 'b')
    plt.xlabel('Freq (Hz)')
    plt.ylabel('Amplitude |X(freq)|')
    plt.title(r'Amplitude spectrum of Time Series $s_1(t) + s_2(t)$')
    plt.xlim(0., .125)

In [ ]:
interactive_plot = interactive(cos_freq_tau, T1=(0.,40.,1.), T2=(0.,160.,1.), tau=(0.,16.,1.))
output = interactive_plot.children[-1]
output.layout.height = '550px'
interactive_plot

How does the time shift affect the two frequencies? Using the sum theorem, we get:

\begin{equation}
s_1(t)+s_2(t)= 2 \cos \left(\omega_1 t +\omega_1\frac{\tau}{2}\right)\, \cos \left(\omega_1\frac{\tau}{2}\right)+2 \cos \left(\omega_2 t +\omega_2\frac{\tau}{2}\right)\, \cos\left(\omega_2\frac{\tau}{2}\right).\tag{5.17}
\end{equation}

- As expected, the frequencies $\omega_1$ and $\omega_2$ are preserved, since the addition of two signals is a linear operation.

- A phase shift occurs at both frequencies, corresponding to a time shift of $\tau/2$. 

- $\cos \left(\omega_1\frac{\tau}{2} \right)=a_1$ and $ \cos\left(\omega_2\frac{\tau}{2}\right)=a_2$ are time-independent weighting factors, but they differ for the two frequencies. 

- Since $\omega_1>\omega_2$, $a_1<a_2$, i.e., the combination is a low-pass filter. The condition for constructive interference is therefore more likely to be met for low frequencies than for high frequencies.

In [ ]:
tau = 10.
w1 = 2. * np.pi / 20.
a1 = np.cos(w1*tau/2.)
print('a1 = ',a1)

In [ ]:
tau = 10.
w2 = 2. * np.pi / 80.
a2 = np.cos(w2*tau/2.)
print('a2 = ',a2)

# Example 4

As an example of a nonlinear transformation of the signal $s(t) = \cos(\omega_0 t)$, consider $s^2(t)$. The resulting signal is 
\begin{equation}
s^2(t) = \frac{1}{2} + \frac{1}{2} \cos(2 \omega_0 t).
\end{equation}  
In contrast to a linear transformation, a new (double) frequency is created.

In [ ]:
def nonlinear(T0, square):
    
    # Define parameters
    dt = .1                  # time sampling [s]
    L = 180.                 # length of the time series [s]

    # Define sine function ...
    t = np.arange(0.,L+dt,dt)      # compute time vector
    
    omega0 = 2. * np.pi / T0      # compute circular frequency from period 1
    
    s1 = np.cos(omega0*t) # compute cos wave 1

    if(square==True):
        s1 = s1**2
        
    s = s1
    
    # Fourier transform
    S = np.fft.fft(s)
    
    # Normalize by length of the time series
    N = len(s)               # length of the time series [#samples]
    S = S / (N*dt)
    
    # estimate frequencies    
    freq = np.fft.fftfreq(N, d=dt)
    
    # Initialize Plots
    plt.figure(figsize=(20,10))
    
    # Plot time series
    plt.subplot(211)
    
    if(square==False):
        plt.plot(t, s1, 'b',label=r'$s(t) = \cos(\omega_0 t)$')
    if(square==True):
        plt.plot(t, s1, 'b',label=r'$s(t) = \cos^2(\omega_0 t)$')
    plt.legend(loc='upper right')
    plt.xlabel('Time (s)')
    plt.ylabel('f(t) (s)')
    plt.xlim(0.,180)
    if(square==False):
        plt.title(r'Time Series $s(t)$')
    if(square==True):
        plt.title(r'Time Series $s^2(t)$')
        
    # Plot Spectrum
    plt.subplot(212)
    
    plt.plot(freq, np.abs(S), 'b')
    plt.xlabel('Freq (Hz)')
    plt.ylabel('Amplitude |X(freq)|')
    plt.title(r'Amplitude spectrum of Time Series $s(t)$')
    plt.xlim(0., .125)

In [ ]:
interactive_plot = interactive(nonlinear, T0=(0.,40.,1.), square=False)
output = interactive_plot.children[-1]
output.layout.height = '600px'
interactive_plot

## Summary:

- Stacking signals can improve the signal-to-noise ratio. 
- Phase differences between harmonic signals result in constructive and destructive interference in the stacked signal.
- Frequency differences in the signals result in frequency modulation of the stacked time series.
- Time delay differences in the time series can lead to attenuation of spectral lines.
- A non-linear transformation of a cosine function leads to a constant bias and a doubling of the frequency.